# Exam Template — Clustering + Evaluation

Checklist tiap soal clustering (image atau text, sama saja setelah jadi angka):
1. Load data -> `X` (array numerik), opsional `y_true` kalau ada label asli buat interpretasi
2. Scale (+ PCA kalau dimensi tinggi / mau divisualisasikan)
3. Tentukan k (elbow + silhouette)
4. Fit clustering final + evaluasi (Silhouette, Davies-Bouldin)
5. Visualisasi (scatter PCA 2D)
6. Insight per cluster (crosstab / describe / purity)
7. Kesimpulan tertulis

Cuma isi bagian yang ada `# TODO`. Sisanya jalanin apa adanya.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

try:
    from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
    HAS_YB = True
except ImportError:
    HAS_YB = False  # fallback manual di section 3 kalau yellowbrick belum terinstall

np.random.seed(42)

## 1. Load data

In [ ]:
# TODO: load dataset sesuai soal, isi salah satu pola di bawah lalu hapus yang tidak dipakai.

# --- pola A: dataset bawaan sklearn (image) ---
# from sklearn.datasets import load_digits
# d = load_digits()
# X_raw, y_true = d.data, d.target

# --- pola B: CSV (tabular / fitur sudah diekstrak) ---
# df = pd.read_csv("nama_file.csv")
# y_true = df.pop("label_column") if "label_column" in df.columns else None  # kalau ada kolom label
# X_raw = df.select_dtypes(include=np.number).values

# --- pola C: folder image (jpg/png per subfolder kelas) ---
# from pathlib import Path
# from PIL import Image
# root = Path("nama_folder")
# paths, labels = [], []
# for ci, class_dir in enumerate(sorted(p for p in root.iterdir() if p.is_dir())):
#     for f in class_dir.iterdir():
#         paths.append(f); labels.append(ci)
# images = np.stack([np.asarray(Image.open(p).convert("RGB").resize((64, 64))) for p in paths])
# X_raw = images.reshape(images.shape[0], -1) / 255.0
# y_true = np.array(labels)

# --- pola D: text (TF-IDF) ---
# from sklearn.feature_extraction.text import TfidfVectorizer
# texts = df["text_column"]
# X_raw = TfidfVectorizer(max_features=500, stop_words="english").fit_transform(texts).toarray()
# y_true = None

X_raw = None   # TODO
y_true = None  # TODO (boleh None kalau tidak ada label asli)

print(f"Shape: {X_raw.shape if X_raw is not None else 'belum diisi'}")

## 2. Preprocessing
Scale wajib. PCA opsional tapi bantu banget kalau dimensi tinggi (image/text) — sekalian dipakai buat plot 2D nanti.

In [ ]:
X_scaled = StandardScaler().fit_transform(X_raw)

N_PC = min(50, X_scaled.shape[0] - 1, X_scaled.shape[1])  # TODO: ganti kalau mau komponen lain
pca = PCA(n_components=N_PC, random_state=42)
X = pca.fit_transform(X_scaled)
print(f"Variance explained ({N_PC} PC): {pca.explained_variance_ratio_.sum():.1%}")

## 3. Tentukan k (Elbow + Silhouette)

In [ ]:
if HAS_YB:
    KElbowVisualizer(KMeans(random_state=42, n_init=10), k=(2, 15), metric="silhouette").fit(X).show()
else:
    k_range = range(2, 15)
    inertias, silhouettes = [], []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X, km.labels_))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(list(k_range), inertias, "o-")
    axes[0].set_title("Elbow Method"); axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
    axes[1].plot(list(k_range), silhouettes, "o-", color="orange")
    axes[1].set_title("Silhouette Score"); axes[1].set_xlabel("k"); axes[1].set_ylabel("Score")
    plt.tight_layout()
    plt.show()

## 4. Clustering final + evaluasi

In [ ]:
K = 5  # TODO: isi sesuai hasil elbow/silhouette (atau jumlah kelas asli kalau diketahui)

model = KMeans(n_clusters=K, random_state=42, n_init=10)
# alternatif kalau KMeans kurang cocok (cluster bukan bulat / jumlah cluster ga pasti):
# model = AgglomerativeClustering(n_clusters=K)
# model = DBSCAN(eps=0.5, min_samples=5)                 # tidak perlu K, tapi perlu tuning eps
# model = GaussianMixture(n_components=K, random_state=42)

cluster_labels = model.fit_predict(X)

sil = silhouette_score(X, cluster_labels)
dbi = davies_bouldin_score(X, cluster_labels)
ch = calinski_harabasz_score(X, cluster_labels)
print(f"Silhouette Score      : {sil:.3f}  (makin dekat 1 makin baik)")
print(f"Davies-Bouldin Index   : {dbi:.3f}  (makin dekat 0 makin baik)")
print(f"Calinski-Harabasz      : {ch:.1f}  (makin tinggi makin baik)")

if y_true is not None:
    ari = adjusted_rand_score(y_true, cluster_labels)
    print(f"Adjusted Rand Index    : {ari:.3f}  (bonus, vs label asli, 1 = cocok sempurna)")

## 5. Visualisasi cluster (PCA 2D)

In [ ]:
plt.figure(figsize=(7, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap="tab10", s=12)
plt.legend(*scatter.legend_elements(), title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title(f"Hasil Clustering ({type(model).__name__}, k={K})")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout()
plt.show()

## 6. Insight per cluster
Kalau ada `y_true` (label asli): crosstab + purity. Kalau tidak: describe statistik tiap fitur per cluster.

In [ ]:
if y_true is not None:
    summary = pd.crosstab(cluster_labels, y_true)
    summary.index.name = "Cluster"
    display(summary)

    for c in range(K):
        if c in summary.index:
            row = summary.loc[c]
            dominant, purity = row.idxmax(), row.max() / row.sum()
            print(f"Cluster {c}: didominasi '{dominant}' (purity {purity:.0%}, n={row.sum()})")
    overall_purity = summary.max(axis=1).sum() / summary.values.sum()
    print(f"\nOverall cluster purity: {overall_purity:.1%}")
else:
    df_feat = pd.DataFrame(X_raw)
    df_feat["cluster"] = cluster_labels
    display(df_feat.groupby("cluster").mean())
    print(df_feat["cluster"].value_counts().sort_index())

## 7. Kesimpulan

<!-- TODO: isi narasi insight tiap cluster di sini. Pancingan:
- Cluster mana paling "bersih"/terpisah (silhouette tinggi lokal, purity tinggi)? kenapa?
- Cluster mana campur/overlap? fitur apa yang bikin mirip?
- Apa makna domain tiap cluster (bukan cuma angka)?
-->